# RAG System Demo: End-to-End Demonstration

This notebook provides a complete demonstration of the Document Question Answering system built with LlamaIndex. It covers:

1. **Document Loading & Preprocessing**
2. **Index Building** with different chunking strategies
3. **Retrieval** and context selection
4. **Generation** with different LLM backends
5. **Evaluation** and result visualization

## 1. Setup and Imports

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import logging
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Imports successful!")

## 2. Load Documents

In [ ]:
from src.loaders import DocumentLoader

# Initialize loader
loader = DocumentLoader('data')

# Load all documents
result = loader.load_all_documents()

print(f"Loaded {len(result.documents)} documents")
print(f"Failed files: {len(result.failed_files)}")
print(f"\nAvailable documents:")
for doc in result.documents:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

## 3. Build Index

In [ ]:
from src.indexing import ChunkingConfig, IndexBuilder

# Choose configuration
config_path = '../configs/chunking_256.yaml'

# Load configuration
config = ChunkingConfig(config_path)

print(f"Configuration:")
print(f"  Chunk size: {config.chunk_size}")
print(f"  Overlap: {config.overlap_ratio}")
print(f"  Strategy: {config.strategy}")
print(f"  Embedding: {config.embedding_model}")

In [ ]:
# Build index
builder = IndexBuilder(config, run_id='demo_run')
index = builder.build_index(result.documents)

print(f"\nIndex built successfully!")
print(f"Output directory: {builder.output_dir}")

## 4. Initialize RAG Pipeline

In [ ]:
from src.rag_pipeline import RAGPipeline, RAGConfig

# Create configuration
rag_config = RAGConfig(
    llm_backend='mock',  # Use mock for demo; change to 'flant5' or 'mistral7b'
    top_k=3,
    max_new_tokens=256,
    temperature=0.1
)

# Create pipeline
pipeline = RAGPipeline(index, rag_config)

print("RAG Pipeline initialized!")

## 5. Run Single Query Demo

In [ ]:
# Run a sample query
query = "What is retrieval-augmented generation (RAG)?"

print(f"Query: {query}")
print("-" * 60)

# Execute query
response = pipeline.query(query)

# Display results
print(f"\n=== ANSWER ===")
print(response.answer)

print(f"\n=== LATENCY ===")
print(f"Retrieval: {response.retrieval_latency:.3f}s")
print(f"Generation: {response.generation_latency:.3f}s")
print(f"Total: {response.total_latency:.3f}s")

## 6. Display Retrieved Chunks

In [ ]:
print("=== RETRIEVED CHUNKS ===")
print(f"Retrieved {len(response.retrieved_chunks)} chunks:\n")

for i, chunk in enumerate(response.retrieved_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(f"ID: {chunk['chunk_id']}")
    print(f"Source: {chunk['source_file']}")
    print(f"Relevance Score: {chunk['score']:.4f}")
    print(f"Text Preview: {chunk['text_preview'][:150]}...")
    print()

## 7. Batch Query Evaluation

In [ ]:
from src.eval_protocol import EvaluationProtocol

# Load evaluation queries
protocol = EvaluationProtocol()
queries = protocol.load_queries('../data/queries.jsonl')

print(f"Loaded {len(queries)} evaluation queries")

# Process a few queries
sample_queries = queries[:5]
results = []

for q in sample_queries:
    print(f"\nProcessing: {q.question[:50]}...")
    response = pipeline.query(q.question)
    results.append({
        'query_id': q.id,
        'query': q.question,
        'answer': response.answer,
        'total_latency': response.total_latency,
        'num_chunks': len(response.retrieved_chunks)
    })

print(f"\nProcessed {len(results)} queries")

## 8. Results Summary

In [ ]:
import pandas as pd

# Create results DataFrame
df = pd.DataFrame(results)
df['answer_length'] = df['answer'].apply(len)

print("=== RESULTS SUMMARY ===")
print(df[['query_id', 'answer_length', 'total_latency', 'num_chunks']].to_string(index=False))

print(f"\nAverage latency: {df['total_latency'].mean():.3f}s")
print(f"Average answer length: {df['answer_length'].mean():.0f} chars")

## 9. Visualization

In [ ]:
import matplotlib.pyplot as plt

# Create visualization directory
Path('../results/figures').mkdir(parents=True, exist_ok=True)

# Create comparison data (simulated for demo)
configs = ['Mistral7B\n256', 'Mistral7B\n512', 'FlanT5\n256', 'FlanT5\n512']
relevance_scores = [4.2, 3.9, 3.5, 3.3]
completion_rates = [0.95, 0.90, 0.80, 0.75]
latencies = [2.5, 3.0, 0.8, 1.2]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Relevance comparison
axes[0].bar(configs, relevance_scores, color=['#2ecc71', '#27ae60', '#3498db', '#2980b9'])
axes[0].set_ylabel('Avg Relevance (1-5)')
axes[0].set_title('Response Relevance')
axes[0].set_ylim(0, 5)
axes[0].axhline(y=4, color='r', linestyle='--', alpha=0.5, label='Target: 4')
axes[0].legend()

# Completion rate comparison
axes[1].bar(configs, completion_rates, color=['#2ecc71', '#27ae60', '#3498db', '#2980b9'])
axes[1].set_ylabel('Task Completion Rate')
axes[1].set_title('Task Completion')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='Target: 90%')
axes[1].legend()

# Latency comparison
axes[2].bar(configs, latencies, color=['#2ecc71', '#27ae60', '#3498db', '#2980b9'])
axes[2].set_ylabel('Latency (seconds)')
axes[2].set_title('Response Latency')

plt.tight_layout()
plt.savefig('../results/figures/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved to ../results/figures/comparison_chart.png")

## 10. Detailed Query Analysis

In [ ]:
# Run a detailed analysis on a single query
detailed_query = "Explain the difference between BM25 and dense retrieval"

print(f"Detailed Query: {detailed_query}")
print("=" * 60)

response = pipeline.query(detailed_query)

print("\n[ANSWER]")
print(response.answer)

print("\n[CITATIONS]")
for cid in response.cited_chunk_ids:
    print(f"  - {cid}")

print("\n[RETRIEVED CONTEXT]")
for i, chunk in enumerate(response.retrieved_chunks, 1):
    print(f"\n[{i}] {chunk['chunk_id']} (score: {chunk['score']:.4f})")
    print(f"    Source: {chunk['source_file']}")
    print(f"    Text: {chunk['text_preview'][:200]}...")

---

## Conclusion

This notebook demonstrated:

- Document loading and preprocessing
- Index building with configurable chunking strategies
- Retrieval-augmented generation pipeline
- Evaluation and result analysis

To run full experiments, use the CLI scripts:

```bash
# Build indices
python src/run_build_index.py --config configs/chunking_256.yaml
python src/run_build_index.py --config configs/chunking_512.yaml

# Run evaluations
python src/run_eval.py --llm flant5 --chunk 256
python src/run_eval.py --llm flant5 --chunk 512
python src/run_eval.py --llm mistral7b --chunk 256
python src/run_eval.py --llm mistral7b --chunk 512
```